Seasonal Effect On bcycle Usage In Oslo City

**This assignment is about detecting how seasons effects the usage of bcycles in Oslo City and difference between weekdays and weekends.**

1. (i) Data Source: https://oslobysykkel.no/en/open-data/historical 
*The data is provided according to the Norwegian License for Open Government Data 2.0 NLOD 2.0.

(ii) There are csv files for every month of the previous years up to now. Each has thousands of datas, with information of starting and ending dates of usage of bicycles, so this provides also duration time, stations ids,names,latitude,longitudes where journey started and ended. Descriptions for starting stations and ending stations.

I probably will need to use stations starting end ending points along duration of usage of bicycle,and dates of the journey so that can help to detect usage according the seasons, and distinguish for weekdays and weekends usage diference.

2. I would like to use table firstly to organize datas that I am going to use and display them. Timeseries line could be also useful: Date on the x-axis, daily trip count on the y-axis so this plot can show season effects on the amount of journey. 

3. My aim is to test two things: whether the level of bicycle usage differs significantly across seasons, and whether the weekday–weekend difference is itself constant across seasons or changes with them.The analysis will start with descriptive statistics and visual exploration and time series of daily trips, box plots of daily volume by season and day type. Since the sample is large, I will report effect sizes as well as p-values, and check the model's assumptions with residual diagnostics before drawing conclusions. I may even be able to predict bicycle necessity and amount according season for next year as well.

In [1]:
import glob
import pandas as pd
files = sorted(glob.glob('Datas/*.csv')) # List all CSV files in the Datas directory
print(len(files)) # Print the number of CSV files found

12


In [2]:
parts = [] # List to hold the DataFrames for each CSV file
for file in files:
    part = pd.read_csv(file) # Read the CSV file into a DataFrame
    parts.append(part) # Append the DataFrame to the list
df = pd.concat(parts, ignore_index=True) # Concatenate all DataFrames into a single DataFrame
print(df.head()) # Print the first few rows of the combined DataFrame

                         started_at                          ended_at  \
0  2025-01-01 04:37:39.287000+00:00  2025-01-01 04:49:08.139000+00:00   
1  2025-01-01 04:44:32.058000+00:00  2025-01-01 04:51:45.817000+00:00   
2  2025-01-01 05:06:52.481000+00:00  2025-01-01 05:11:28.880000+00:00   
3  2025-01-01 05:22:23.200000+00:00  2025-01-01 05:28:45.298000+00:00   
4  2025-01-01 06:05:12.184000+00:00  2025-01-01 06:08:39.238000+00:00   

   duration  start_station_id     start_station_name  \
0       688               441        Vippetangen øst   
1       433               535       Uranienborgveien   
2       276               521     Jess Carlsens gate   
3       382               480  Helga Helgesens plass   
4       207               507      Jens Bjelkes Gate   

  start_station_description  start_station_latitude  start_station_longitude  \
0           nærmest Operaen               59.903989                10.741077   
1     vest for Slottsparken               59.920042             

In [3]:
df.shape

(1113904, 13)

In [4]:
df["started_at"] = pd.to_datetime(df["started_at"], format="ISO8601") # Convert the 'started_at' column to datetime format
df["ended_at"] = pd.to_datetime(df["ended_at"], format="ISO8601") # Convert the 'ended_at' column to datetime format

In [5]:
df.dtypes

started_at                   datetime64[ns, UTC]
ended_at                     datetime64[ns, UTC]
duration                                   int64
start_station_id                           int64
start_station_name                        object
start_station_description                 object
start_station_latitude                   float64
start_station_longitude                  float64
end_station_id                             int64
end_station_name                          object
end_station_description                   object
end_station_latitude                     float64
end_station_longitude                    float64
dtype: object

In [6]:
df["started_at"] = df["started_at"].dt.tz_convert("Europe/Oslo")
df["ended_at"]   = df["ended_at"].dt.tz_convert("Europe/Oslo")  

In [7]:
# i need to supervise the data to decrease and use only the information i need. 
df = df.drop(columns=["start_station_id", "start_station_name", "end_station_id", "end_station_name","start_station_description","start_station_latitude","start_station_longitude","end_station_description","end_station_latitude","end_station_longitude"]) 
# Drop unnecessary columns

In [8]:
df.columns

Index(['started_at', 'ended_at', 'duration'], dtype='object')

In [9]:
df.dtypes.head(2)

started_at    datetime64[ns, Europe/Oslo]
ended_at      datetime64[ns, Europe/Oslo]
dtype: object

In [10]:
df[df["started_at"].dt.month == 12].head(5)

,started_at,ended_at,duration
1089579,2025-12-01 00:06:28.965000+01:00,2025-12-01 00:17:06.196000+01:00,637
1089580,2025-12-01 00:22:27.269000+01:00,2025-12-01 00:28:07.560000+01:00,340
1089581,2025-12-01 00:36:38.058000+01:00,2025-12-01 00:41:19.958000+01:00,281
1089582,2025-12-01 00:40:02.735000+01:00,2025-12-01 00:42:33.269000+01:00,150
1089583,2025-12-01 00:43:30.106000+01:00,2025-12-01 01:29:08.194000+01:00,2738


In [11]:
df["date"] = df["started_at"].dt.date # Create a new column 'date' with only the date part of 'started_at'

In [12]:
df.shape #tested if new column is added

(1113904, 4)

In [13]:
df.loc[1089579, "date"] 
#testing the date for the row 1089579 which seemed to be in november in UTC Timezone, but it should be in December for Oslo Timezone. The date is correct, it is in December for Oslo Timezone.

datetime.date(2025, 12, 1)

In [14]:
# will add day_type column for weekday and weekend analysis.
import numpy as np
#pandas gives the day of the week as an integer, where Monday is 0 and Sunday is 6.
df["day_type"] = np.where(df["started_at"].dt.dayofweek < 5, "Weekday", "Weekend")
df["day_type"].value_counts() # Check the distribution of weekday and weekend rides

day_type
Weekday    894320
Weekend    219584
Name: count, dtype: int64

In [15]:
df.groupby("day_type")["date"].nunique() # Count the number of unique dates for each day type (Weekday vs Weekend)

day_type
Weekday    261
Weekend    104
Name: date, dtype: int64

in this result it seems during weekdays there are more bcycle usage than weekends. 
weekday average per day: 894320/261= 3426.51
weekend average per day: 219584/104= 2111.38

comparison between weekdays and weekend: 3426.51/2111.38= 1.62
Note: In Oslo bcycle usage per weekday is 62% more than per weekend.
The raw total ratio (894,320 / 219,584 ≈ 4.07) is misleading, because 2025 has 2.5 times more weekdays than weekend days.

In [16]:
# i need to add season column to the data to analyze the data by season.
# Winter: December, January, February are 12,1,2 
# Spring: March, April, May are 3,4,5
# Summer: June, July, August are 6,7,8
# Autumn: September, October, November are 9,10,11

season_map={12: "Winter", 1: "Winter", 2: "Winter",
            3: "Spring", 4: "Spring", 5: "Spring",
            6: "Summer", 7: "Summer", 8: "Summer",
            9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["season"] = df["started_at"].dt.month.map(season_map)
df["season"].value_counts() # Check the distribution of rides across seasons

season
Summer    471161
Autumn    308028
Spring    284564
Winter     50151
Name: count, dtype: int64

In [17]:
df.groupby("season")["date"].nunique() # Count the number of unique dates for each season

season
Autumn    91
Spring    92
Summer    92
Winter    90
Name: date, dtype: int64

In [18]:
df["season"].isna().sum()

np.int64(0)